# Replication of vanilla RNN with numpy.

### Model is created for prediction of the next element of the sequence based on previous element

At time step $t$, the vanilla RNN updates are:

$$
h_t = \tanh(x_t \cdot W_{xh} + W_{hh} \cdot h_{t-1} + b_h)
$$

$$
y_t = W_{hy} \cdot h_t + b_y
$$


In [3]:
import torch

In [15]:
# Creating the class with forward and backward propagations
class RNN():
    def __init__(self):
        # Initializing params
        # self.W_xh = torch.randn((1,5), requires_grad=True)
        # self.W_hh = torch.randn((5,5), requires_grad=True)
        # self.b_h = torch.randn((5), requires_grad=True)
        # self.W_hy = torch.randn((1,5), requires_grad=True)
        # self.b_y = torch.randn((1), requires_grad=True)
        # WARNING - fancy initializing of the weights, why the value is just stuck if not initialize like that?
        import torch.nn.init as init 
        self.W_xh = torch.empty(1, 50).requires_grad_(True)
        self.W_hh = torch.empty(50, 50).requires_grad_(True)
        self.W_hy = torch.empty(1, 50).requires_grad_(True)
        init.xavier_uniform_(self.W_xh)
        init.xavier_uniform_(self.W_hh)
        init.xavier_uniform_(self.W_hy)
        self.b_h = torch.zeros(50).requires_grad_(True)
        self.b_y = torch.zeros(1).requires_grad_(True)

        # Making the default initial hidden state
        self.h_init = torch.ones((50,), requires_grad=True)  # WARNING - why shape is not (5,5)?
    
    # Uses "n" element to predict "n+1" element
    def forward(self, sequence):
        # The MSE loss
        self.loss = 0
        # Counter of accurate predictions
        acc_counter = []
        # Making the initialized hidden state as prior h_t 
        h_t = self.h_init
        for i in range(len(sequence)):
            x_input = sequence[i].unsqueeze(0) # WARNING - to make it as 1d, form factor
            h_t = torch.tanh(x_input@self.W_xh + self.W_hh@h_t + self.b_h)
            y_t = self.W_hy@h_t+self.b_y
            print(y_t) # WARNING - somehow I get nan!
            if i!=len(sequence)-1:
                y_real = sequence[i+1]
                isCorrect = round(float(y_t))==round(float(y_real))
                print(f"Predicted: {float(y_t)} Real: {y_real} IsCorrect: {isCorrect}")
                acc_counter.append(isCorrect)
                # Calculating the loss for this timestep
                loss_t = (y_real-y_t)**2
                self.loss+=loss_t
            else:
                self.loss = self.loss / (len(sequence) - 1) # WARNING - why it is better to divide the loss after?
                return y_t, sum(acc_counter)
    
    # Backpropagation + updating the weights, biases + zeroing the gradients for future
    def backward_and_update(self, lr=0.1):
        self.loss.backward()
        
        # # WARNING - the gradient clipping, what is it and does it help?
        # torch.nn.utils.clip_grad_norm_([self.W_xh, self.W_hh, self.W_hy, self.b_h, self.b_y], max_norm=1.0)
        
        try:
            self.W_xh.data -= lr*self.W_xh.grad
            self.W_hh.data -= lr*self.W_hh.grad
            self.b_h.data -= lr*self.b_h.grad
            self.W_hy.data -= lr*self.W_hy.grad
            self.b_y.data -= lr*self.b_y.grad
            
            # Manually zero gradients
            self.W_xh.grad.zero_()
            self.W_hh.grad.zero_()
            self.b_h.grad.zero_()
            self.W_hy.grad.zero_()
            self.b_y.grad.zero_()
        except TypeError as e:
            print("Caught TypeError:", e)
            print("Developer: probably because there was no forward() called")
        
        

long_sequence = [1,2,4,8,16,32,64,128,256,512,1024,2048]
long_sequence = torch.tensor(long_sequence, dtype=torch.float32)
#long_sequence = torch.log2(long_sequence)
#short_sequence = [1,3,9]
model = RNN()
for i in range(5000):
    prediction, accuracy = model.forward(long_sequence)
    model.backward_and_update()

tensor([-0.3650], grad_fn=<AddBackward0>)
Predicted: -0.36503082513809204 Real: 2.0 IsCorrect: False
tensor([0.0262], grad_fn=<AddBackward0>)
Predicted: 0.026237845420837402 Real: 4.0 IsCorrect: False
tensor([0.0054], grad_fn=<AddBackward0>)
Predicted: 0.005396753549575806 Real: 8.0 IsCorrect: False
tensor([-0.5835], grad_fn=<AddBackward0>)
Predicted: -0.5835028886795044 Real: 16.0 IsCorrect: False
tensor([-0.4593], grad_fn=<AddBackward0>)
Predicted: -0.45933032035827637 Real: 32.0 IsCorrect: False
tensor([-0.5752], grad_fn=<AddBackward0>)
Predicted: -0.5752012729644775 Real: 64.0 IsCorrect: False
tensor([-0.8842], grad_fn=<AddBackward0>)
Predicted: -0.884161651134491 Real: 128.0 IsCorrect: False
tensor([-0.8703], grad_fn=<AddBackward0>)
Predicted: -0.8703400492668152 Real: 256.0 IsCorrect: False
tensor([-0.8628], grad_fn=<AddBackward0>)
Predicted: -0.8627581596374512 Real: 512.0 IsCorrect: False
tensor([-0.8619], grad_fn=<AddBackward0>)
Predicted: -0.86194908618927 Real: 1024.0 IsCorr

ValueError: cannot convert float NaN to integer